# DWD ICON precipitation — direct-HTTPS fetch

DWD Open Data publishes ICON forecasts as per-variable, bz2-compressed
GRIB2 files over plain HTTPS (no SDK, no auth). earthlens' DWD centre
downloads and decompresses them.

**Requirements** (live download):

```bash
pip install earthlens[nwp]
```

!!! note "Native grid"
    DWD's native ICON-global files are on an **icosahedral** grid, which
    is not a regular lat/lon raster — so this notebook fetches the raw
    GRIB2 rather than cropping it to a COG. For a croppable COG use a
    regular-lat/lon ICON product.

In [ ]:
import datetime as dt
from pathlib import Path

from earthlens.nwp import Catalog
from earthlens.nwp.centres.dwd import DWDCentre

model = Catalog().get_model("icon-global")
model.bands

In [ ]:
# Yesterday's 00Z run (DWD keeps only ~the last day online).
cycle = (dt.datetime.now(dt.UTC) - dt.timedelta(days=1)).replace(
    hour=0, minute=0, second=0, microsecond=0, tzinfo=None
)
out_dir = Path("out/icon")
out_dir.mkdir(parents=True, exist_ok=True)

grib_path = DWDCentre(out_dir).fetch_one(
    model, cycle, step=0, params=["precipitation_acc"], mirror="auto"
)
grib_path, grib_path.stat().st_size

The downloaded `.grib2` holds the requested band's decompressed
messages. For a regular-grid model you would instead call
`EarthLens(data_source="nwp", variables={...}).download()` and receive a
bbox-cropped COG, exactly like the GFS quickstart.